In [1]:
from typing import Literal
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv(override=True)
client = Anthropic()
model = "claude-sonnet-4-0"

In [2]:
def add_msg(
    messages: list[dict[str, str]] | None = None,
    role: Literal["user", "assistant"] = "user",
    text: str = None
) -> list[dict[str, str]]:
    if not messages:
        messages = []
    msg: dict[str, str] = {"role": role, "content": text}
    messages.append(msg)
    return messages

def chat(
    messages: list[dict[str, str]],
    system: str | None = None,
    temperature: float = 1.0
) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params.update({"system": system})

    ans = client.messages.create(**params)

    # for k, v in ans.model_dump().items():
    #     print(k, ':', v)

    return ans.content[0].text

In [3]:
messages = add_msg(text="Write a 1 sentence description of a fake database")

In [ ]:
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_018kS7EJiLvTdemRcd7isp63', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=8, server_tool_use=None, service_tier='standard'), stop_details=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text="Here's a one-sentence description of", type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' a fake database:\n\n"PetPersonalities is a fictional database containing detailed', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaE

In [12]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        pass
stream.get_final_message()

"UserProfiles_v2_FINAL" is a corrupted SQLite database containing 50,000 fake user records with randomly generated names, email addresses, and demographic data used for testing the company's new customer analytics dashboard.

ParsedMessage(id='msg_015ddwmkYMjCxD54APZatBs9', container=None, content=[ParsedTextBlock(citations=None, text='"UserProfiles_v2_FINAL" is a corrupted SQLite database containing 50,000 fake user records with randomly generated names, email addresses, and demographic data used for testing the company\'s new customer analytics dashboard.', type='text', parsed_output=None)], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=52, server_tool_use=None, service_tier='standard'), stop_details=None)

In [43]:
messages = add_msg(text="What is quantum computing?")

answer = chat(messages)

messages = add_msg(messages, "assistant", answer)

messages = add_msg(messages=messages, text="Write another sentence")

answer= chat(messages)

answer

'Quantum computing represents one of the most promising frontiers in technology, with the potential to revolutionize fields from medicine to cybersecurity within the next few decades.'

In [46]:
user_msg = input("Hy, How can I help you today?")
print(user_msg)
messages = add_msg(text=user_msg)
while True:
    answer = chat(messages)
    print(answer)
    messages = add_msg(messages, "assistant", answer)
    user_msg = input()
    print(user_msg)
    messages = add_msg(messages=messages, text=user_msg)

What's 1+1?
1+1 = 2
Add 2 to that answer.
2 + 2 = 4
Minus 1.
4 - 1 = 3


KeyboardInterrupt: Interrupted by user

In [19]:
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

messages = add_msg(text="How do I solve 5x+3=2 for x?")

answer = chat(messages, system)
answer

'Great question! Let\'s work through this step by step. \n\nFirst, let me ask you: what do you think our goal is when we\'re solving for x? What do we want x to look like at the end?\n\nOnce you think about that, look at the equation 5x + 3 = 2. What do you notice is "attached" to the x term that we might want to remove first?'

In [21]:
system = """
You are a Python engineer.
You respond only with code, without providing explanations.
"""

messages = add_msg(text="Write a Python function that checks a string for duplicate characters.")

answer = chat(messages, system)

answer

'```python\ndef has_duplicates(s):\n    return len(s) != len(set(s))\n```'

In [25]:
messages = None

messages = add_msg(text="Generate a one sentence movie idea")

answer = chat(messages, temperature=1.0)

answer

'A lonely librarian discovers that the books in her small-town library are portals to the worlds they contain, but each time someone enters a story, they risk being trapped forever as a character in that narrative.'